# Check start/end points head with coarse Naches ATS simulation

- **check the start/end points head for selected v4 Hillslope 1**
- generate a BC-head figure for review before saving the selected hillslope for the following workflow


Head extracted from Zhi's 3D ATS simulation for Naches

- file `global/cfs/cdirs/m1800/naches_run2_share/Naches-2`
- Information
    - "Time": from 11224 to 16059; unit is day;
        - 11400 = 85+365x31 --> 2011.03.26
        - 12631 = 221+365x34 --> 2014.08.09

Output of this script

- review head at the selected Hillslope 1 starting and end points from Naches-2
- save the selected v4 hillslope coordinates and metadata for the following notebooks


In [ ]:
%load_ext autoreload
%autoreload 2

## config Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
case_config = config['case']
spinup_config = config['spinup']
prefire_transient_config = config['prefire_transient']
watershed_name = case_config['watershed_name']
hucs = [case_config['hucs']]
site_name = case_config['site_name']

# simulation control
start_year_spinup = int(spinup_config['source_period']['start_date'][:4])
end_year_spinup = int(spinup_config['source_period']['end_date'][:4])
nyears_steadystate_spinup = spinup_config['steady_state_years']
nyears_cyclic_spinup = spinup_config['cyclic_years']
start_year_transient = int(prefire_transient_config['start_date'][:4])
end_year_transient = int(prefire_transient_config['end_date'][:4])

In [ ]:
outputs={}

## extract point raw data from 3D ATS simulation

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd
import rasterio

import ats_xdmf as xdmf
import time
import random
import pandas
import os

from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import geopandas as gpd

In [ ]:
## get stream network
# Essential imports for watershed workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.plot
import pyproj

# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)

# Set up watershed workflow CRS (DayMet CRS)
crs_daymet = watershed_workflow.crs.daymet_crs()

# Set up sources
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']

# Parameters for river extraction
hucs_config = [case_config['hucs']]
ignore_small_rivers = 2
prune_by_area_fraction = 0.0

# Get HUC12 list
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
    return huc12_list

hucs = get_huc12(hucs_config)
huc_level = 12

print(f"Processing HUCs: {hucs[:5]}...")

# Load watershed HUCs
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs_daymet)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

# Download/collect the river network
print("Downloading river network...")
_, reaches = watershed_workflow.get_reaches(sources['hydrography'], hucs[0], 
                                            watershed.exterior(), crs_daymet, crs_daymet,
                                            in_network=True, properties=True)

# Construct river network
rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                             ignore_small_rivers=ignore_small_rivers,
                                             prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                             remove_diversions=True,
                                             remove_braided_divergences=True)

print(f"Number of rivers: {len(rivers)}")

### Config 3D ATS-flow results

In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
model_dir = '/global/cfs/cdirs/m1800/naches_run2_share/Naches-2'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

flag_atsptextract = True

### Load the selected v4 hillslope

This replaces the legacy list of hillslopes larger than 100 m. The v4 candidate CSV is generated by 0a-transect_latlon.Naches.v4.D8.ipynb; this notebook reviews only Hillslope 1.


In [ ]:
# Load one selected hillslope from the v4 candidate CSV
selected_hillslope_id = 'Hillslope 1'
csv_filename = f'../data-processed/{site_name}/hillslope_candidates_{site_name}_v4.csv'
df = pd.read_csv(csv_filename)

required_columns = {'hillslope_id', 'start_x', 'start_y', 'end_x', 'end_y', 'hillslope_length_m'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Candidate CSV is missing columns: {sorted(missing_columns)}')

selected_rows = df.loc[df['hillslope_id'] == selected_hillslope_id]
if len(selected_rows) != 1:
    raise ValueError(f'Expected exactly one {selected_hillslope_id!r}, found {len(selected_rows)}')
df = selected_rows.reset_index(drop=True)

# Retain the variable names used by the unchanged legacy head-plotting cells.
start_coords_all = df[['start_x', 'start_y']].values
end_coords_all = df[['end_x', 'end_y']].values
hillslope_names = df['hillslope_id'].tolist()
point_ids = [f"{df.iloc[0].get('endpoint_label_a', 'v4')}/{df.iloc[0].get('endpoint_label_b', 'v4')}"]
hillslope_ids = df['hillslope_id'].tolist()
distances = df['hillslope_length_m'].values
num_transects = len(df)

print(f"Loaded selected {selected_hillslope_id} from: {csv_filename}")
display(df)


### Load surface/subsurface h5 files

In [ ]:
## a function to estimate WTD based on pressure

def get_ats_wtd_pressurebased(pressure_subsurface,visfile_surface, visfile_subsurface):
    # visfile_subsurface.centroids.shape -> (n_surface, 14, 3); 14 is soil layers, bottom-top
    iz_coord = visfile_subsurface.centroids[:,:,-1]
    
    ### Find Equivalent Surface and Subsurface ID based on cell centroids
    # take some care on the rounding of coordinates, can cause errors
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    # [remarks] prepare to perform a pairwise compare [n_surface,1,2] with [1,n_surface,2] -> returns [n_surface, n_surface]
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    assert surface_subsurface_IDs[:,1].shape == visfile_surface.centroids[:,1].shape, f"Shape mismatch: change the round precision in the surface/subsurface centroid coordinates"
    
    ### Estimate WTD based on pressure
    # pressure head
    ih = (pressure_subsurface - patm) / (rho * g) # dim=(time, n_surface, 14)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1) # dim=(time, n_surface)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    # WTD elevation
    # [remark] iH_rev=part1+part2; part1 is the relative distance between water table and the first saturated subsurface cell
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
    wtdep_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    pwdep_pressure_based = - np.minimum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    assert pressure_subsurface[:,:,1].shape == wtdep_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"
    
    ### Re-arrange WTD in accordance to surface IDs
    # As pressure obtained from subsurface does not follow surface ID, re-arrange this:
    head_rearranged  = head_pressure_based[:, subsurface_indices]
    wtdep_rearranged = wtdep_pressure_based[:, subsurface_indices]
    pwdep_rearranged = pwdep_pressure_based[:, subsurface_indices]
    
    return surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged

In [ ]:
if flag_atsptextract:
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    print(visfile_surface.times)

    tmp_year = 2026 # a standard year
    day_number0 = int(visfile_surface.times[0])
    date0 = datetime.strptime(f'{tmp_year}-{day_number0%365}', '%Y-%j')
    print(f"t0: {day_number0}={day_number0%365}+365x{day_number0//365}")
    print(f"t0: {day_number0%365} is {date0.strftime('%B %d')}")
    
    day_number1 = int(visfile_surface.times[-1])
    date1 = datetime.strptime(f'{tmp_year}-{day_number1%365}', '%Y-%j')
    print(f"t1: {day_number1}={day_number1%365}+365x{day_number1//365}")
    print(f"t1: {day_number1%365} is {date1.strftime('%B %d')}")
    
else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # subsurface
    start = time.time()
    visfile_subsurface = xdmf.VisFile(directory=model_dir,
                                      filename="ats_vis_data.h5", 
                                      mesh_filename="ats_vis_mesh.h5")
    visfile_subsurface.loadMesh(columnar=True)
    end = time.time()
    print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")
    
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    # subsurface pressure (time, xy-space and soil columns)
    start = time.time()
    pressure_subsurface = visfile_subsurface.getArray('pressure')
    end = time.time()
    print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")
    
    # get the WTD (based on surface ATS ID)
    start = time.time()
    surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                                    visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
    end = time.time()
    print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

else:
    print("Skipping: data already extracted")

### Diagnose 3D endpoint data and cut the selected v4 hillslope

The outlet cut uses a fine-DEM local convention: \(H_{\mathrm{cut}}=z_{\mathrm{NHD}}+\min(h_{3D})\).  It is a practical shoreline-cut level, not an absolute 3D water-surface reconstruction.  A coarse outlet triangle can mix river and land, so its centroid elevation is saved only as QC and is never added to \(h_{3D}\) for this cut.

After locating the continuous shoreline, the ridge-to-outlet distance is rounded upward to a whole metre toward the NHD side.  The applied outlet series is \(h_{\mathrm{outlet,2D}}=h_{3D}-(z_{\mathrm{out}}-z_{\mathrm{NHD}})\), so its low-stage value is near zero while the 3D fluctuation is retained.  This notebook only diagnoses and saves cut geometry; production BC series remain the responsibility of the `get_BChead` notebooks.  The start groundwater-depth diagnostic remains relative to local ground.


In [ ]:
if False:  # Legacy start/end head plot; BC construction occurs in get_BChead notebooks.
    surface_x_coord = visfile_surface.centroids[:,0]
    surface_y_coord = visfile_surface.centroids[:,1]

    surface_times = visfile_surface.times
    print(f"Number of timesteps: {len(surface_times)}")

    # The legacy plotting logic is retained, but v4 has exactly one selected transect.
    idx = 0
    start_coords = (start_coords_all[idx, 0], start_coords_all[idx, 1])
    end_coords = (end_coords_all[idx, 0], end_coords_all[idx, 1])

    dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
    dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)
    start_index = np.argmin(dist_start)
    end_index = np.argmin(dist_end)
    min_dist_start = dist_start[start_index]
    min_dist_end = dist_end[end_index]

    startpt_head = head_rearranged[:, start_index]
    endpt_head = head_rearranged[:, end_index]

    print(f"\nHillslope: {hillslope_names[idx]}")
    print(f"  Endpoints: {point_ids[idx]}")
    print(f"  Start point - Index: {start_index}, Distance: {min_dist_start:.2f}m")
    print(f"  End point - Index: {end_index}, Distance: {min_dist_end:.2f}m")

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(surface_times, startpt_head, color='blue', linewidth=2, label='Start point head')
    ax.plot(surface_times, endpt_head, color='red', linewidth=2, label='End point head')
    ax.set_xlabel('Time [days]')
    ax.set_ylabel('Head [m]')
    ax.set_title(f'Boundary head for {hillslope_names[idx]}\nEndpoints: {point_ids[idx]}, Distance: {distances[idx]:.1f}m')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

    plot_dir = Path(f'./images/{site_name}')
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_filename = plot_dir / f'{hillslope_names[idx].replace(" ", "_")}.bchead.png'
    plt.tight_layout()
    plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
    #plt.close()
    print(f"  Saved plot to: {plot_filename}")
else:
    print("Skipping: data already extracted")


In [ ]:
# Cut the raw transect at the fine-DEM low-stage shoreline, then align its length to metres.
# Treat min(h_3D) as a persistent coarse-mesh offset above the local NHD DEM elevation.
if not flag_atsptextract:
    raise RuntimeError('Run the 3D ATS extraction cells before constructing the cut BC artifact.')

raw_start = np.asarray(start_coords_all[0], dtype=float)
raw_nhd = np.asarray(end_coords_all[0], dtype=float)
raw_length = float(np.linalg.norm(raw_nhd - raw_start))
if raw_length <= 0:
    raise ValueError('The selected raw transect has zero length.')
direction = (raw_nhd - raw_start) / raw_length

# Match points to containing 3D surface triangles, not merely their centroids.
etype, surface_nodes, surface_conn = xdmf.meshXYZ(model_dir, 'ats_vis_surface_mesh.h5')
surface_conn = np.asarray(surface_conn, dtype=int)
if surface_conn.min() == 1:
    surface_conn -= 1
surface_triangles = [Polygon(np.asarray(surface_nodes)[nodes, :2]) for nodes in surface_conn]

surface_times = np.asarray(visfile_surface.times)
surface_ponded_depth = np.asarray(visfile_surface.getArray('surface-ponded_depth'))

def containing_surface_index(point, label, prefer_wettest=False):
    matches = [i for i, triangle in enumerate(surface_triangles) if triangle.covers(Point(point))]
    if not matches:
        raise ValueError(f'{label} is outside the 3D surface mesh.')
    if len(matches) > 1:
        if not prefer_wettest:
            raise ValueError(f'{label} is on a 3D cell edge; select the containing land cell explicitly.')
        return max(matches, key=lambda index: surface_ponded_depth[:, index].mean())
    return matches[0]

start_index = containing_surface_index(raw_start, 'Ridge start')
outlet_index = containing_surface_index(raw_nhd, 'Raw NHD outlet', prefer_wettest=True)
outlet_ponded_depth_3d = np.asarray(surface_ponded_depth[:, outlet_index])
if outlet_ponded_depth_3d.min() < 0:
    raise ValueError(f'Expected non-negative ponded depth, got {outlet_ponded_depth_3d.min():.6g} m.')
delta_z_low_stage = float(outlet_ponded_depth_3d.min())
# A coarse triangle may span river and land; retain its centroid elevation for QC only.
outlet_centroid_z = float(visfile_surface.centroids[outlet_index, -1])

dem_path = Path('./data/dem/reprojected_dem.tif')
if not dem_path.exists():
    raise FileNotFoundError(f'DEM not found: {dem_path}. Run 0a-transect_latlon.Naches.v4.D8.ipynb first.')
def bilinear_dem_sample(dem_src, points):
    """Sample band 1 at map coordinates using bilinear interpolation."""
    points = np.asarray(points, dtype=float)
    cols, rows = ~dem_src.transform * (points[:, 0], points[:, 1])
    cols = np.asarray(cols) - 0.5
    rows = np.asarray(rows) - 0.5
    col0 = np.floor(cols).astype(int)
    row0 = np.floor(rows).astype(int)
    if np.any((col0 < 0) | (row0 < 0) | (col0 + 1 >= dem_src.width) | (row0 + 1 >= dem_src.height)):
        raise ValueError('A requested point is too close to or outside the DEM boundary for bilinear sampling.')
    dx = cols - col0
    dy = rows - row0
    corners = np.column_stack([
        np.repeat(col0, 4) + np.tile([0, 1, 0, 1], len(points)) + 0.5,
        np.repeat(row0, 4) + np.tile([0, 0, 1, 1], len(points)) + 0.5,
    ])
    corner_xy = np.asarray([dem_src.transform * tuple(corner) for corner in corners])
    values = np.asarray([sample[0] for sample in dem_src.sample(corner_xy)]).reshape(-1, 4)
    if not np.isfinite(values).all() or (dem_src.nodata is not None and np.any(values == dem_src.nodata)):
        raise ValueError('A requested point has an invalid DEM value.')
    return ((1 - dx) * (1 - dy) * values[:, 0] + dx * (1 - dy) * values[:, 1]
            + (1 - dx) * dy * values[:, 2] + dx * dy * values[:, 3])

profile_s = np.arange(0.0, np.ceil(raw_length) + 1.0, 1.0)
profile_s[-1] = raw_length
profile_xy = raw_start + profile_s[:, None] * direction
with rasterio.open(dem_path) as dem_src:
    profile_z = bilinear_dem_sample(dem_src, profile_xy)
if not np.isfinite(profile_z).all():
    raise ValueError('The raw transect extends outside the valid DEM.')
z_nhd = float(profile_z[-1])
# This is a constructed local cut level, not an absolute 3D water-surface elevation.
low_stage_level = z_nhd + delta_z_low_stage
high_stage_level = z_nhd + float(outlet_ponded_depth_3d.max())

# Only the NHD-connected flooded toe is removed; equality is wet at low stage.
flooded = profile_z <= low_stage_level
if not flooded[-1]:
    raise ValueError('The raw NHD endpoint is not below the low-stage shoreline.')
last_dry_index = len(profile_s) - 1
while last_dry_index > 0 and flooded[last_dry_index]:
    last_dry_index -= 1
if last_dry_index == 0:
    raise ValueError('The low-stage shoreline would remove the entire transect.')
first_wet_index = last_dry_index + 1
z_dry, z_wet = profile_z[last_dry_index], profile_z[first_wet_index]
if z_dry <= low_stage_level or z_wet > low_stage_level:
    raise RuntimeError('Could not bracket the NHD-connected low-stage shoreline.')
crossing_fraction = (low_stage_level - z_dry) / (z_wet - z_dry)
L_cut_raw = float(profile_s[last_dry_index] + crossing_fraction * (profile_s[first_wet_index] - profile_s[last_dry_index]))
# Distance increases from ridge to NHD, so ceil() moves the outlet to the wet/NHD side.
L_cut = float(np.ceil(L_cut_raw))
if L_cut > raw_length:
    raise ValueError('Rounded cut length extends beyond the original NHD endpoint.')
final_outlet_xy = raw_start + L_cut * direction
provisional_outlet_xy = raw_start + L_cut_raw * direction
with rasterio.open(dem_path) as dem_src:
    z_start, z_out = bilinear_dem_sample(dem_src, np.vstack([raw_start, final_outlet_xy]))
if not np.isfinite([z_start, z_out]).all():
    raise ValueError('The rounded endpoint has an invalid DEM elevation.')
if z_out > low_stage_level + 1.e-6:
    raise ValueError('The rounded outlet is not on the NHD-connected low-stage wet side.')
# Shift the depth series by the fine-DEM elevation change from NHD to the rounded outlet.
# The wet-side check above makes min(h_3D - delta_z_BC) non-negative, apart from tolerance.
delta_z_BC = float(z_out - z_nhd)
start_triangle_pressure_head = np.asarray(head_rearranged[:, start_index])
revised_outlet_ponded_depth = outlet_ponded_depth_3d - delta_z_BC
if revised_outlet_ponded_depth.min() < -1.e-6:
    raise ValueError('The rounded outlet produces a negative revised ponded depth.')
revised_outlet_ponded_depth = np.maximum(revised_outlet_ponded_depth, 0.0)
revised_outlet_same_triangle = bool(surface_triangles[outlet_index].covers(Point(final_outlet_xy)))
high_stage_flooded = profile_z <= high_stage_level
high_stage_last_dry = len(profile_s) - 1
while high_stage_last_dry > 0 and high_stage_flooded[high_stage_last_dry]:
    high_stage_last_dry -= 1
if high_stage_last_dry == 0:
    high_stage_wet_length_m = raw_length
else:
    high_stage_first_wet = high_stage_last_dry + 1
    high_fraction = ((high_stage_level - profile_z[high_stage_last_dry])
                     / (profile_z[high_stage_first_wet] - profile_z[high_stage_last_dry]))
    high_shoreline_s = profile_s[high_stage_last_dry] + high_fraction * (profile_s[high_stage_first_wet] - profile_s[high_stage_last_dry])
    high_stage_wet_length_m = raw_length - float(high_shoreline_s)

start_coarse_z = float(visfile_surface.centroids[start_index, -1])
cut_geometry = {
    'raw_start_xy': raw_start, 'raw_nhd_xy': raw_nhd,
    'provisional_outlet_xy': provisional_outlet_xy,
    'final_outlet_xy': final_outlet_xy,
    'raw_length_m': raw_length, 'L_cut_raw_m': L_cut_raw, 'L_cut_m': L_cut,
    'rounding_increment_m': L_cut - L_cut_raw,
    'z_nhd_m': z_nhd, 'z_start_m': z_start, 'z_out_m': z_out,
    'low_stage_level_m': low_stage_level, 'high_stage_level_m': high_stage_level,
    'delta_z_low_stage_m': delta_z_low_stage, 'delta_z_BC_m': delta_z_BC,
    'high_stage_wet_length_m': high_stage_wet_length_m,
    'start_surface_cell': start_index, 'outlet_surface_cell': outlet_index,
    'revised_outlet_same_triangle': revised_outlet_same_triangle,
    'start_coarse_z_m': start_coarse_z,
    'outlet_triangle_centroid_x_m': float(visfile_surface.centroids[outlet_index, 0]),
    'outlet_triangle_centroid_y_m': float(visfile_surface.centroids[outlet_index, 1]),
    'outlet_triangle_centroid_z_m': float(visfile_surface.centroids[outlet_index, 2]),
    'start_coarse_minus_local_z_m': start_coarse_z - z_start,
}

cut_summary = pd.DataFrame([{
    **{key: value for key, value in cut_geometry.items() if np.isscalar(value)},
    'raw_start_x': raw_start[0], 'raw_start_y': raw_start[1],
    'raw_nhd_x': raw_nhd[0], 'raw_nhd_y': raw_nhd[1],
    'provisional_outlet_x': provisional_outlet_xy[0],
    'provisional_outlet_y': provisional_outlet_xy[1],
    'final_outlet_x': final_outlet_xy[0], 'final_outlet_y': final_outlet_xy[1],
    'hillslope_end_x_m': final_outlet_xy[0], 'hillslope_end_y_m': final_outlet_xy[1],
    'hillslope_end_z_m': z_out,
}])
cut_summary_path = Path(f'../data-processed/{site_name}/cut_transect_{site_name}_summary.csv')
cut_summary.to_csv(cut_summary_path, index=False)
fig, ax_profile = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax_profile.plot(profile_s, profile_z, color='black', label='Fine DEM')
ax_profile.axhline(low_stage_level, color='tab:blue', linestyle='--', label='Local low stage: z_NHD + min(h_3D)')
ax_profile.axhline(high_stage_level, color='tab:cyan', linestyle=':', label='Local high stage: z_NHD + max(h_3D)')
ax_profile.axvline(L_cut_raw, color='tab:orange', linestyle='--', label='Provisional shoreline')
ax_profile.axvline(L_cut, color='tab:red', label='Rounded outlet')
ax_profile.set(xlabel='Distance from ridge [m]', ylabel='Elevation [m]', title='Shoreline cut and rounded outlet')
ax_profile.legend(); ax_profile.grid(alpha=.3)
qc_path = Path(f'./images/{site_name}/cut_transect_bc_qc.png')
qc_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(qc_path, dpi=150, bbox_inches='tight')
plt.show()

fig_bc, ax_bc = plt.subplots(figsize=(12, 6))
ax_bc.plot(surface_times, start_triangle_pressure_head, color='tab:blue', label='Start triangle: 3D pressure head')
ax_bc.plot(surface_times, outlet_ponded_depth_3d, color='tab:red', label='Original outlet triangle: raw h_3D')
ax_bc.plot(surface_times, revised_outlet_ponded_depth, color='tab:green', label='Cut outlet: h_3D - delta_z_BC')
ax_bc.set(xlabel='Time [days]', ylabel='Head / ponded depth [m]',
          title='Nested 3D endpoint diagnostics and cut-outlet adjustment')
ax_bc.legend(loc='best'); ax_bc.grid(alpha=.3)
bc_plot_path = Path(f'./images/{site_name}/{hillslope_names[0].replace(" ", "_")}.cut_bchead.png')
fig_bc.savefig(bc_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"L_cut_raw={L_cut_raw:.3f} m; L_cut={L_cut:.0f} m; rounding={L_cut - L_cut_raw:.3f} m")
print(f"z_centroid (QC only)={outlet_centroid_z:.3f} m; z_NHD={z_nhd:.3f} m; z_out={z_out:.3f} m")
print(f"min(h_3D)={delta_z_low_stage:.3f} m; local low_stage={low_stage_level:.3f} m; local high_stage={high_stage_level:.3f} m; delta_z_BC={delta_z_BC:.3f} m")
print(f"Provisional outlet={provisional_outlet_xy}; final outlet={final_outlet_xy}")
print(f"Revised outlet in the original 3D outlet triangle: {revised_outlet_same_triangle}")
print(f"High-stage NHD-connected wet length={high_stage_wet_length_m:.3f} m")
print(f"Saved geometry-only cut metadata to {cut_summary_path}, {qc_path}, and {bc_plot_path}")


## Save user selected hillslopes

In [ ]:
# ==============================================================================
# USER INPUT: Select the v4 hillslope to save
# ==============================================================================
# v4 already identifies the desired transect. Keep this list structure so the
# following legacy multi/single save cells retain their original workflow.
selected_hillslopes = [
    {'hillslope_id': 'Hillslope 1', 'name': 'Hillslope_1'}
]
# ==============================================================================

print("\n" + "="*100)
print("SAVING SELECTED HILLSLOPE TRANSECTS")
print("="*100)

df_all_hillslopes = df.copy()
transects_to_save = []

# The v4 D8 notebook writes this NED DEM in DayMet CRS, which matches the
# candidate endpoint coordinates.
dem_path = Path('./data/dem/reprojected_dem.tif')
if not dem_path.exists():
    raise FileNotFoundError(f'DEM not found: {dem_path}. Run 0a-transect_latlon.Naches.v4.D8.ipynb first.')

with rasterio.open(dem_path) as dem_src:
    for idx, selection in enumerate(selected_hillslopes):

        hillslope_id = selection['hillslope_id']
        custom_name = selection.get('name', hillslope_id)
        matching_rows = df_all_hillslopes.loc[df_all_hillslopes['hillslope_id'] == hillslope_id]
        if len(matching_rows) != 1:
            raise ValueError(f'Expected one selected row for {hillslope_id!r}, found {len(matching_rows)}')

        row = matching_rows.iloc[0]
        start_coords = cut_geometry['raw_start_xy']
        end_coords = cut_geometry['final_outlet_xy']
        distance = cut_geometry['L_cut_m']
        start_z = cut_geometry['z_start_m']
        end_z = cut_geometry['z_out_m']
        elev_drop = start_z - end_z

        transects_to_save.append({
            'name': custom_name,
            'point_id': row.get('endpoint_label_a', 'v4'),
            'hillslope_id': hillslope_id,
            'neighbor_idx': np.nan,
            'start_coords': start_coords,
            'start_z': start_z,
            'end_coords': end_coords,
            'end_z': end_z,
            'distance': distance,
            'elev_drop': elev_drop,
            'raw_end_coords': cut_geometry['raw_nhd_xy'],
            'provisional_end_coords': cut_geometry['provisional_outlet_xy'],
            'raw_length': cut_geometry['raw_length_m'],
            'L_cut_raw': cut_geometry['L_cut_raw_m'],
            'rounding_increment': cut_geometry['rounding_increment_m'],
            'z_nhd': cut_geometry['z_nhd_m'],
            'outlet_centroid_z': cut_geometry['outlet_triangle_centroid_z_m'],
            'low_stage_level': cut_geometry['low_stage_level_m'],
            'high_stage_level': cut_geometry['high_stage_level_m'],
            'delta_z_low_stage': cut_geometry['delta_z_low_stage_m'],
            'delta_z_BC': cut_geometry['delta_z_BC_m']
        })

        print(f"\n[{idx+1}] {custom_name}")
        print(f"  Start: ({start_coords[0]:.2f}, {start_coords[1]:.2f}), Z={start_z:.1f}m")
        print(f"  End:   ({end_coords[0]:.2f}, {end_coords[1]:.2f}), Z={end_z:.1f}m")
        print(f"  Distance: {distance:.1f}m, Elev Drop: {elev_drop:.1f}m")


In [ ]:
# ==============================================================================
# USER INPUT: Select ONE hillslope to save as single transect
# ==============================================================================
# 
# Select the index (0-based) of the hillslope from transects_to_save
# For example: 
#   0 = first hillslope
#   1 = second hillslope
#   etc.

selected_single_index = 0  # Hillslope 1 is the only v4 selection

# ==============================================================================

if len(transects_to_save) > 0:
    from scipy.io import savemat
    
    # Validate index
    if selected_single_index < 0 or selected_single_index >= len(transects_to_save):
        print(f"\n✗ ERROR: Invalid index {selected_single_index}")
        print(f"   Valid range: 0 to {len(transects_to_save)-1}")
        print(f"   Available transects:")
        for i, t in enumerate(transects_to_save):
            print(f"     [{i}] {t['name']}")
    else:
        # Get selected transect
        selected_transect = transects_to_save[selected_single_index]
        
        print("\n" + "="*100)
        print("SAVING SINGLE HILLSLOPE TRANSECT")
        print("="*100)
        print(f"\nSelected: [{selected_single_index}] {selected_transect['name']}")
        print(f"  Point ID: {selected_transect['point_id']}")
        print(f"  Hillslope ID: {selected_transect['hillslope_id']}")
        print(f"  Start: ({selected_transect['start_coords'][0]:.2f}, {selected_transect['start_coords'][1]:.2f}), Z={selected_transect['start_z']:.1f}m")
        print(f"  End:   ({selected_transect['end_coords'][0]:.2f}, {selected_transect['end_coords'][1]:.2f}), Z={selected_transect['end_z']:.1f}m")
        print(f"  Distance: {selected_transect['distance']:.1f}m")
        print(f"  Elev Drop: {selected_transect['elev_drop']:.1f}m")
        
        # Prepare data for single transect (compatible with original format)
        single_transect_data = {
            'start_coords': selected_transect['start_coords'],
            'end_coords': selected_transect['end_coords'],
            'metadata': {
                'name': selected_transect['name'],
                'point_id': selected_transect['point_id'],
                'hillslope_id': selected_transect['hillslope_id'],
                'neighbor_idx': selected_transect['neighbor_idx'],
                'distance': selected_transect['distance'],
                'elev_drop': selected_transect['elev_drop'],
                'start_elevation': selected_transect['start_z'],
                'end_elevation': selected_transect['end_z'],
                'raw_end_coords': selected_transect['raw_end_coords'],
                'provisional_end_coords': selected_transect['provisional_end_coords'],
                'raw_length': selected_transect['raw_length'],
                'L_cut_raw': selected_transect['L_cut_raw'],
                'rounding_increment': selected_transect['rounding_increment'],
                'z_nhd': selected_transect['z_nhd'],
                'outlet_centroid_z': selected_transect['outlet_centroid_z'],
                'low_stage_level': selected_transect['low_stage_level'],
                'high_stage_level': selected_transect['high_stage_level'],
                'delta_z_low_stage': selected_transect['delta_z_low_stage'],
                'delta_z_BC': selected_transect['delta_z_BC']
            },
            'selected_category': site_name
        }
        
        # Save to MAT file
        single_mat_filename = f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
        savemat(single_mat_filename, single_transect_data)
        
        print(f"\n✓ Saved single transect to: {single_mat_filename}")
        print(f"\nFile contents:")
        print(f"  - start_coords: [2] array")
        print(f"  - end_coords: [2] array")
        print(f"  - metadata: Dictionary with transect details")
        
        # Also save a human-readable CSV
        csv_filename = f'../data-processed/{site_name}/startendcoords_{site_name}_summary.csv'
        summary_df = pd.DataFrame({
            'transect_name': [selected_transect['name']],
            'point_id': [selected_transect['point_id']],
            'hillslope_id': [selected_transect['hillslope_id']],
            'neighbor_idx': [selected_transect['neighbor_idx']],
            'start_x': [selected_transect['start_coords'][0]],
            'start_y': [selected_transect['start_coords'][1]],
            'start_z': [selected_transect['start_z']],
            'end_x': [selected_transect['end_coords'][0]],
            'end_y': [selected_transect['end_coords'][1]],
            'end_z': [selected_transect['end_z']],
            'distance_m': [selected_transect['distance']],
            'elev_drop_m': [selected_transect['elev_drop']],
            'raw_end_x': [selected_transect['raw_end_coords'][0]],
            'raw_end_y': [selected_transect['raw_end_coords'][1]],
            'provisional_end_x': [selected_transect['provisional_end_coords'][0]],
            'provisional_end_y': [selected_transect['provisional_end_coords'][1]],
            'raw_length_m': [selected_transect['raw_length']],
            'L_cut_raw_m': [selected_transect['L_cut_raw']],
            'rounding_increment_m': [selected_transect['rounding_increment']],
            'z_nhd_m': [selected_transect['z_nhd']],
            'outlet_centroid_z_m': [selected_transect['outlet_centroid_z']],
            'low_stage_level_m': [selected_transect['low_stage_level']],
            'high_stage_level_m': [selected_transect['high_stage_level']],
            'delta_z_low_stage_m': [selected_transect['delta_z_low_stage']],
            'delta_z_BC_m': [selected_transect['delta_z_BC']]
        })
        summary_df.to_csv(csv_filename, index=False)
        print(f"✓ Also saved summary CSV: {csv_filename}")
        
else:
    print("\n⚠️  WARNING: No transects available to save!")
    print("   Please run the previous cell to select and save multiple transects first.")